<a href="https://colab.research.google.com/github/ishivansmishra/ChatBot/blob/main/Chatbots_With_Langraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
!pip install langgraph langsmith

In [69]:
!pip install langchain langchain_groq langchain_community

In [70]:
from google.colab import userdata
groq_api_key = userdata.get('GROQ_API_KEY')
langsmith_key = userdata.get('langsmith_key')

In [71]:
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "CourseLanggraph"
os.environ["LANGCHAIN_API_KEY"] = langsmith_key

In [72]:
from langchain_groq import ChatGroq


In [74]:
llm = ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.6'}}, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7faf02e86480>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7faf0299e330>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
## Start Building Chatbot using langgraph

In [75]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

In [76]:
class State(TypedDict):
  messages:Annotated[list,add_messages]
graph_builder = StateGraph(State)


In [77]:
graph_builder

In [78]:
def chatbot(state:State):
  return {"messages":llm.invoke(state['messages'])}

In [79]:
graph_builder.add_node("chatbot",chatbot)

In [80]:
graph_builder

In [81]:
graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("chatbot",END)

In [82]:
graph = graph_builder.compile()

In [ ]:
while True:
  user_input = input("User: ")
  if user_input.lower() in ["quit","q"] :
    print("Good Bye")
    break
  for event in graph.stream({'messages':('user',user_input)}):
    print(event.values())
    for value in event.values():
      print(value['messages'])
      print("Assistant: ",value["messages"].content)

User: hello
dict_values([{'messages': AIMessage(content='Hello. How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 36, 'total_tokens': 46, 'completion_time': 0.010153848, 'completion_tokens_details': None, 'prompt_time': 0.00226653, 'prompt_tokens_details': None, 'queue_time': 0.060230716, 'total_time': 0.012420378}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ecbb2-d0f4-7493-965a-867d94eb8710-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 10, 'total_tokens': 46})}])
content='Hello. How can I assist you today?' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 36, 'total_tokens': 46, 'completion_time': 0.010153848, 'completion_tokens_details': None, 'prompt_time': 0.0022